<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dinosaur Training Notebook

Train a dinosaur to walk, run, and perform species-specific behaviors using reinforcement learning.

**Supported Species:**
- **Velociraptor** — Bipedal predator with sickle-claw strike
- **T-Rex** — Large bipedal predator with bite attack
- **Brachiosaurus** — Quadrupedal sauropod with food reaching

**Training Stages (all species):**
1. **Balance** — Learn to stand without falling
2. **Locomotion** — Walk and run forward
3. **Species behavior** — Strike / Bite / Food reach

**Supported Algorithms:** PPO, SAC

Change the `SPECIES` variable in the configuration cell below to select which dinosaur to train.
All configs are loaded from the TOML files in `configs/<species>/`.

## 1. Setup & Installation

In [ ]:
# Install dependencies (Colab auto-detected; no-op locally)
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system(
            'pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib'
        )
    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally.")

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path (works both locally from notebooks/ and in Colab)
if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

from environments.shared.config import load_all_stages, save_stage_config
from environments.shared.curriculum import (
    RewardRampCallback,
    SaveVecNormalizeCallback,
    StageWarmupCallback,
    load_vecnorm_stats,
)
from environments.shared.evaluation import evaluate

# Library imports — shared training infrastructure
from environments.shared.train_base import (
    SpeciesConfig,
    create_vec_env,
    linear_schedule,
)

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

Change `SPECIES` below to train a different dinosaur. All other settings apply to every species.

In [ ]:
# ===== SPECIES SELECTION =====
# Change this to: "velociraptor", "trex", or "brachiosaurus"
SPECIES = "velociraptor"

# ===== Training settings =====
ALGORITHM = "ppo"      # "ppo" or "sac"
N_ENVS = 4             # Number of parallel environments
SEED = 42              # Random seed for reproducibility
VERBOSE = 0            # 0=quiet, 1=progress bar, 2=debug
QUICK_TEST = False     # True = tiny run to verify setup works
USE_GOOGLE_DRIVE = True  # Mount Google Drive to save results

print(f"Species: {SPECIES}")
print(f"Algorithm: {ALGORITHM.upper()}")
print(f"Quick test mode: {QUICK_TEST}")

In [ ]:
# ============================================================
# Storage Configuration
# ============================================================
# When USE_GOOGLE_DRIVE is True and running in Colab, logs and
# models are saved to Google Drive so they persist across sessions.
# Otherwise, everything is saved to the local filesystem.

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOG_BASE = Path("/content/drive/MyDrive/mesozoic-labs/logs")
    LOG_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Logs will be saved to: {LOG_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")
else:
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")

# Create a single run directory for all stages, organised by species then datetime
RUN_DIR = LOG_BASE / SPECIES / f"{ALGORITHM.lower()}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {RUN_DIR}")

## 3. Explore the Environment

In [ ]:
# Resolve species environment class and build SpeciesConfig
import importlib

_SPECIES_MAP = {
    "velociraptor": {
        "module": "environments.velociraptor.envs.raptor_env",
        "class": "RaptorEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=strike",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["strike_success", "bite_success"],
    },
    "trex": {
        "module": "environments.trex.envs.trex_env",
        "class": "TRexEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=bite",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["bite_success", "strike_success"],
    },
    "brachiosaurus": {
        "module": "environments.brachiosaurus.envs.brachio_env",
        "class": "BrachioEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=food_reach",
        "height_label": "Torso height",
        "stage3_section_label": "Food Reaching",
        "success_keys": ["food_reached"],
    },
}

assert SPECIES in _SPECIES_MAP, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_MAP.keys())}"
_info = _SPECIES_MAP[SPECIES]
_mod = importlib.import_module(_info["module"])
EnvClass = getattr(_mod, _info["class"])

# Build SpeciesConfig (same structure used by the CLI train scripts)
SPECIES_CFG = SpeciesConfig(
    species=SPECIES,
    env_class=EnvClass,
    stage_descriptions=_info["stage_descriptions"],
    height_label=_info["height_label"],
    stage3_section_label=_info["stage3_section_label"],
    success_keys=_info["success_keys"],
)

# Load all stage configs from TOML files
STAGE_CONFIGS = load_all_stages(SPECIES)

env = EnvClass()
print(f"Environment: {EnvClass.__name__}")
print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.shape}")
print(f"Number of actuators: {env.model.nu}")
for stage_num, cfg in STAGE_CONFIGS.items():
    print(f"  Stage {stage_num}: {cfg['name']} - {cfg['description']}")
env.close()

In [ ]:
# Run random episodes to establish a baseline
env = EnvClass()
n_episodes = 5
episode_rewards = []
episode_lengths = []

for ep in range(n_episodes):
    obs, info = env.reset(seed=ep)
    total_reward = 0
    step = 0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step += 1
        if terminated or truncated:
            break
    episode_rewards.append(total_reward)
    episode_lengths.append(step)
    print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

print("\nRandom policy baseline:")
print(f"  Avg reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
print(f"  Avg length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
env.close()

## 4. Training Infrastructure

In [ ]:
import time

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback, EvalCallback

from environments.shared.diagnostics import DiagnosticsCallback
from environments.shared.evaluation import eval_policy
from environments.shared.reporting import (
    build_stage_results_from_eval_data,
    generate_stage_artifacts,
    save_results_csv as _lib_save_results_csv,
    save_results_json as _lib_save_results_json,
)
from environments.shared.reporting import (
    write_training_summary as _lib_write_training_summary,
)

ALGO_CLASS = {"PPO": PPO, "SAC": SAC}

try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Videos will be skipped. Install with: pip install mediapy")

# Accumulates (stage_num, stage_dir) tuples as each stage completes,
# so training curves can be plotted incrementally after every stage.
completed_stages = []


def _eval_forward_vel(model, stage, vecnorm_path, n_episodes=30):
    """Evaluate a trained policy collecting forward velocity, distance, and success rate.

    Thin wrapper around the library's ``eval_policy`` that handles
    environment creation and VecNormalize loading using notebook globals.
    """
    eval_env = create_vec_env(SPECIES_CFG, STAGE_CONFIGS, stage, 1, SEED + 3000)
    if vecnorm_path and Path(vecnorm_path).exists():
        load_vecnorm_stats(vecnorm_path, eval_env)
    eval_env.training = False
    eval_env.norm_reward = False

    rewards, lengths, fwd_vels, successes, distances = eval_policy(
        model, eval_env, SPECIES_CFG.success_keys, n_episodes=n_episodes,
    )
    eval_env.close()
    return rewards, lengths, fwd_vels, successes, distances


def display_stage_videos(stage, stage_dir):
    """Display saved stage videos inline using mediapy.

    Finds all ``.mp4`` files in *stage_dir* and plays them inline.
    This is used after ``generate_stage_artifacts`` has already recorded
    the videos to disk.
    """
    if not _HAS_MEDIAPY:
        print(f"Skipping video display for stage {stage} (mediapy not installed).")
        return
    stage_dir = Path(stage_dir)
    mp4s = sorted(stage_dir.glob("*.mp4"))
    if not mp4s:
        print(f"No videos found in {stage_dir}")
        return
    for mp4 in mp4s:
        print(f"Playing: {mp4.name}")
        mediapy.show_video(mediapy.read_video(str(mp4)), fps=50)


def train_stage(
    stage,
    timesteps,
    load_path=None,
    run_dir=None,
    vecnorm_path=None,
):
    """Train a single curriculum stage.

    Uses library functions for environment creation (``create_vec_env``),
    VecNormalize transfer (``load_vecnorm_stats``), and LR scheduling
    (``linear_schedule``).  The notebook-specific ``DiagnosticsCallback``
    is added on top of the standard SB3 callbacks.

    Returns (model, best_model_path, final_model_path, stage_dir, vecnorm_save_path, stage_results).
    """
    stage_start = time.time()

    config = STAGE_CONFIGS[stage]
    AlgoClass = ALGO_CLASS[ALGORITHM.upper()]

    # Algorithm hyperparameters from TOML config
    algo_key = "sac_kwargs" if ALGORITHM.lower() == "sac" else "ppo_kwargs"
    algo_kwargs = config[algo_key].copy()
    algo_kwargs["verbose"] = VERBOSE

    # Extract policy_kwargs from TOML config, falling back to default net_arch
    policy_kwargs = algo_kwargs.pop("policy_kwargs", {"net_arch": [256, 256]})

    # Handle learning_rate_end: create a linear schedule using the library helper
    lr_end = algo_kwargs.pop("learning_rate_end", None)
    if lr_end is not None:
        lr_start = algo_kwargs["learning_rate"]
        algo_kwargs["learning_rate"] = linear_schedule(lr_start, lr_end)

    # Directories - each stage gets a subdirectory within the run directory
    if run_dir is None:
        run_dir = repo_root / "logs"
    stage_dir = Path(run_dir) / f"stage{stage}"
    stage_dir.mkdir(parents=True, exist_ok=True)
    model_dir = stage_dir / "models"
    model_dir.mkdir(exist_ok=True)
    algo_kwargs["tensorboard_log"] = str(stage_dir / "tensorboard")

    print(f"{'=' * 60}")
    print(f"Stage {stage}: {config['name']} ({ALGORITHM})")
    print(f"Description: {config['description']}")
    print(f"Timesteps: {timesteps:,}")
    print(f"Log dir: {stage_dir}")
    if vecnorm_path:
        print(f"VecNormalize from prior stage: {vecnorm_path}")
    print(f"{'=' * 60}")

    # Resolve stage transition parameters up front so the exact values
    # (whether from TOML or defaults) are recorded in the saved config.
    cur_kwargs = config.get("curriculum_kwargs", {})
    extra_meta = {"seed": SEED, "n_envs": N_ENVS, "timesteps": timesteps}
    if stage > 1:
        warmup_timesteps = cur_kwargs.get("warmup_timesteps", 100_000)
        warmup_clip_range = cur_kwargs.get("warmup_clip_range", 0.02)
        warmup_ent_coef = cur_kwargs.get("warmup_ent_coef", 0.02)
        ramp_timesteps = cur_kwargs.get("ramp_timesteps", 500_000)
        ramp_start_value = cur_kwargs.get("ramp_start_value", 0.1)
        target_fwd_weight = config["env_kwargs"].get("forward_vel_weight", 1.0)
        extra_meta.update({
            "warmup_timesteps": warmup_timesteps,
            "warmup_clip_range": warmup_clip_range,
            "warmup_ent_coef": warmup_ent_coef,
            "ramp_timesteps": ramp_timesteps,
            "ramp_start_value": ramp_start_value,
            "forward_vel_weight": target_fwd_weight,
        })

    # Save reward weights, hyperparameters, and resolved stage transition
    # params for reproducibility.  The "run" section captures the actual
    # values used (including defaults) so a run can be replicated exactly.
    cfg_path = save_stage_config(
        stage_dir,
        stage,
        config,
        ALGORITHM,
        extra=extra_meta,
        env_class=EnvClass,
    )
    print(f"Stage config saved to: {cfg_path}")

    # Create environments using library infrastructure.
    # SAC benefits from SubprocVecEnv (parallel env stepping) since it
    # interleaves gradient updates with env steps in a tight loop.
    # PPO collects large rollout batches so DummyVecEnv overhead is negligible.
    use_subproc = ALGORITHM.lower() == "sac" and N_ENVS > 1
    if use_subproc:
        print(f"Using SubprocVecEnv for SAC ({N_ENVS} parallel workers)")
    train_env = create_vec_env(SPECIES_CFG, STAGE_CONFIGS, stage, N_ENVS, SEED, use_subproc=use_subproc)
    eval_env = create_vec_env(SPECIES_CFG, STAGE_CONFIGS, stage, 1, SEED + 1000)

    # Load VecNormalize stats from prior stage using library
    if vecnorm_path:
        if not load_vecnorm_stats(vecnorm_path, train_env, eval_env):
            print(f"WARNING: VecNormalize file not found: {vecnorm_path} — eval env will use defaults")
            eval_env.training = False
            eval_env.norm_reward = False
    else:
        # Even without prior stats, the eval env should never update
        # running statistics or normalise rewards during evaluation.
        eval_env.training = False
        eval_env.norm_reward = False

    # Create or load model
    if load_path:
        print(f"Loading model from: {load_path}")
        model = AlgoClass.load(
            load_path,
            env=train_env,
            **algo_kwargs,
        )
    else:
        model = AlgoClass(
            "MlpPolicy",
            train_env,
            policy_kwargs=policy_kwargs,
            **algo_kwargs,
        )

    # Callbacks
    # SaveVecNormalizeCallback fires via callback_on_new_best so the
    # VecNormalize stats are saved alongside best_model.zip, ensuring
    # the obs normalization matches the policy weights.
    best_vecnorm_path = str(model_dir / "best_model_vecnorm.pkl")
    save_vecnorm_cb = SaveVecNormalizeCallback(save_path=best_vecnorm_path)

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(model_dir),
        log_path=str(stage_dir),
        eval_freq=max(50000 // N_ENVS, 1),
        n_eval_episodes=30,
        deterministic=True,
        verbose=max(VERBOSE, 1),
        callback_on_new_best=save_vecnorm_cb,
    )
    checkpoint_callback = CheckpointCallback(
        save_freq=max(250000 // N_ENVS, 1),
        save_path=str(model_dir),
        name_prefix=f"stage{stage}",
        save_vecnormalize=True,
    )
    diagnostics_callback = DiagnosticsCallback(
        plateau_window=10,
        plateau_threshold=1.0,
        log_dir=str(stage_dir),
    )

    # Stage transition callbacks from library (stages 2+): warm up the value
    # function and gradually ramp the forward velocity reward to prevent
    # catastrophic forgetting of balance behaviours.
    stage_transition_cbs = []
    if stage > 1 and ALGORITHM.upper() == "PPO":
        stage_transition_cbs.append(StageWarmupCallback(
            warmup_timesteps=warmup_timesteps,
            warmup_clip_range=warmup_clip_range,
            warmup_ent_coef=warmup_ent_coef,
        ))
    if stage > 1:
        stage_transition_cbs.append(RewardRampCallback(
            attr_name="forward_vel_weight",
            start_value=ramp_start_value,
            end_value=target_fwd_weight,
            ramp_timesteps=ramp_timesteps,
        ))

    # Train
    model.learn(
        total_timesteps=timesteps,
        callback=CallbackList([eval_callback, checkpoint_callback, diagnostics_callback] + stage_transition_cbs),
        progress_bar=VERBOSE >= 1,
    )

    # Save final model and VecNormalize stats
    final_path = model_dir / f"stage{stage}_final"
    model.save(str(final_path))
    final_vecnorm_path = str(final_path) + "_vecnorm.pkl"
    train_env.save(final_vecnorm_path)
    vecnorm_save_path = final_vecnorm_path
    print(f"\nFinal model saved to: {final_path}.zip")
    print(f"VecNormalize stats saved to: {final_vecnorm_path}")

    # Evaluate the final model with its matching VecNormalize stats.
    episode_rewards, episode_lengths, episode_fwd_vels, episode_successes, episode_distances = _eval_forward_vel(
        model, stage, final_vecnorm_path, n_episodes=30,
    )
    mean_reward = float(np.mean(episode_rewards))
    std_reward = float(np.std(episode_rewards))
    mean_length = float(np.mean(episode_lengths))
    std_length = float(np.std(episode_lengths))
    mean_fwd_vel = float(np.mean(episode_fwd_vels))
    std_fwd_vel = float(np.std(episode_fwd_vels))
    mean_distance = float(np.mean(episode_distances))
    sim_dt = float(eval_env.get_attr("dt")[0])
    mean_success_rate = float(np.mean(episode_successes))
    print(f"Eval (final model): mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
    print(f"Eval: mean_length={mean_length:.1f} +/- {std_length:.1f} steps ({mean_length * sim_dt:.2f}s sim time)")
    print(f"Eval: mean_forward_vel={mean_fwd_vel:.2f} +/- {std_fwd_vel:.2f} m/s")
    print(f"Eval: mean_distance_traveled={mean_distance:.2f} m")
    print(f"Eval: success_rate={mean_success_rate:.0%}")

    train_env.close()
    eval_env.close()

    # Build base results dict from on-disk eval data (evaluations.npz),
    # then enrich with the live 30-episode evaluation metrics above.
    # This uses the same shared function that the sweep trial worker uses,
    # ensuring the best-eval-checkpoint metrics are computed identically.
    stage_duration = time.time() - stage_start
    stage_results = build_stage_results_from_eval_data(
        stage_dir, stage, config, timesteps=timesteps,
        duration_seconds=stage_duration,
    )
    best_eval_reward = stage_results["best_eval_reward"]
    best_eval_std = stage_results["best_eval_std"]
    best_eval_length = stage_results["best_eval_length"]
    best_eval_timestep = stage_results["best_eval_timestep"]
    if best_eval_reward != "":
        print(f"Best model eval:  mean_reward={best_eval_reward} +/- {best_eval_std} (at {best_eval_timestep:,} steps)")

    # Override with richer live-eval metrics
    stage_results.update({
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "mean_episode_length": mean_length,
        "std_episode_length": std_length,
        "mean_forward_vel": mean_fwd_vel,
        "std_forward_vel": std_fwd_vel,
        "mean_distance_traveled": mean_distance,
        "mean_success_rate": mean_success_rate,
        "sim_dt": sim_dt,
    })

    # Use best model for video recording and next-stage loading.
    # The matched VecNormalize was saved by SaveVecNormalizeCallback
    # via EvalCallback's callback_on_new_best.
    best_model_zip = model_dir / "best_model.zip"
    best_model_reward, best_model_std_reward = "", ""
    best_model_length, best_model_std_length = "", ""
    best_model_fwd_vel, best_model_std_fwd_vel = "", ""
    best_model_distance = ""
    best_model_success_rate = ""
    if best_model_zip.exists():
        best_path = model_dir / "best_model"
        model = AlgoClass.load(str(best_path))
        output_path = str(best_path)
        print(f"Loaded best model for next-stage: {best_path}.zip")

        if Path(best_vecnorm_path).exists():
            vecnorm_save_path = best_vecnorm_path
            print(f"Using matched VecNormalize for best model: {best_vecnorm_path}")
        else:
            print("best_model_vecnorm.pkl not found; using final VecNormalize for best model.")

        # Evaluate the best model over 30 episodes
        print("Evaluating best model (30 episodes)...")
        bm_rewards, bm_lengths, bm_fwd_vels, bm_successes, bm_distances = _eval_forward_vel(
            model, stage, vecnorm_save_path, n_episodes=30,
        )
        best_model_reward = round(float(np.mean(bm_rewards)), 2)
        best_model_std_reward = round(float(np.std(bm_rewards)), 2)
        best_model_length = round(float(np.mean(bm_lengths)), 1)
        best_model_std_length = round(float(np.std(bm_lengths)), 1)
        best_model_fwd_vel = round(float(np.mean(bm_fwd_vels)), 2)
        best_model_std_fwd_vel = round(float(np.std(bm_fwd_vels)), 2)
        best_model_distance = round(float(np.mean(bm_distances)), 2)
        best_model_success_rate = round(float(np.mean(bm_successes)), 2)
        print(f"Best model eval:  mean_reward={best_model_reward} +/- {best_model_std_reward}")
        print(f"Best model eval:  mean_length={best_model_length} +/- {best_model_std_length}")
        print(f"Best model eval:  mean_fwd_vel={best_model_fwd_vel} +/- {best_model_std_fwd_vel} m/s")
        print(f"Best model eval:  mean_distance={best_model_distance} m")
        print(f"Best model eval:  success_rate={best_model_success_rate:.0%}")
    else:
        output_path = str(final_path)

    # Check curriculum gating thresholds using the best model's metrics.
    # The best model is what gets handed off to the next stage, so the
    # gate decision should reflect *its* performance — not the final
    # model's, which may have degraded late in training.
    # Use best model eval when available, fall back to evaluations.npz
    # best-checkpoint metrics, then final model as last resort.
    gate_reward = float(best_model_reward) if best_model_reward != "" else (float(best_eval_reward) if best_eval_reward != "" else mean_reward)
    gate_length = float(best_model_length) if best_model_length != "" else (float(best_eval_length) if best_eval_length != "" else mean_length)
    gate_fwd_vel = float(best_model_fwd_vel) if best_model_fwd_vel != "" else mean_fwd_vel
    gate_success = float(best_model_success_rate) if best_model_success_rate != "" else mean_success_rate

    target_reward = cur_kwargs.get("min_avg_reward", -float("inf"))
    target_length = cur_kwargs.get("min_avg_episode_length", 0.0)
    target_fwd_vel = cur_kwargs.get("min_avg_forward_vel", 0.0)
    target_success_rate = cur_kwargs.get("min_success_rate", 0.0)

    gate_failures = []
    if gate_reward < target_reward:
        gate_failures.append(f"best model reward {gate_reward:.2f} < {target_reward:.2f}")
    if gate_length < target_length:
        gate_failures.append(f"best model episode length {gate_length:.1f} < {target_length:.1f}")
    if target_fwd_vel > 0.0 and gate_fwd_vel < target_fwd_vel:
        gate_failures.append(f"best model forward vel {gate_fwd_vel:.2f} m/s < {target_fwd_vel:.2f} m/s")
    if target_success_rate > 0.0 and gate_success < target_success_rate:
        gate_failures.append(f"best model success rate {gate_success:.0%} < {target_success_rate:.0%}")

    gate_passed = len(gate_failures) == 0
    if not gate_passed:
        print(
            f"\n*** WARNING: Stage {stage} did not meet curriculum gate: "
            + "; ".join(gate_failures)
            + ". ***"
        )

    # Update stage_results with model paths, gating, and best-model eval
    stage_results.update({
        "model_path": output_path,
        "final_model_path": str(final_path),
        "vecnorm_path": vecnorm_save_path,
        "final_vecnorm_path": final_vecnorm_path,
        "gate_passed": gate_passed,
        "gate_failures": gate_failures,
        "best_model_reward": best_model_reward,
        "best_model_std_reward": best_model_std_reward,
        "best_model_length": best_model_length,
        "best_model_std_length": best_model_std_length,
        "best_model_fwd_vel": best_model_fwd_vel,
        "best_model_std_fwd_vel": best_model_std_fwd_vel,
        "best_model_distance": best_model_distance,
        "best_model_success_rate": best_model_success_rate,
    })

    return model, output_path, str(final_path), stage_dir, vecnorm_save_path, stage_results


def write_training_summary(run_dir, stage_results_list, species=None):
    """Write a training summary text file to the run directory."""
    if species is None:
        species = SPECIES
    path = _lib_write_training_summary(
        run_dir, stage_results_list,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        n_envs=N_ENVS,
        quick_test=QUICK_TEST,
    )
    summary_text = path.read_text()
    print(f"\nTraining summary saved to: {path}")
    print(summary_text)


def save_results_json(stage_results_list, species=SPECIES):
    """Save a summary.json to results/<species>/<algorithm>/ in the repo."""
    algo_lower = ALGORITHM.lower()
    results_dir = repo_root / "results" / species / algo_lower
    path = _lib_save_results_json(
        stage_results_list,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        results_dir=results_dir,
    )
    print(f"\nResults summary saved to: {path}")
    return path


def save_results_csv(stage_results_list, run_dir=None, species=SPECIES):
    """Save a collected_results.csv to the run directory.

    Produces a CSV with the same column structure as the sweep's
    collected_results.csv, containing per-stage rows with all
    hyperparameters, metrics, thresholds, and gate pass/fail status.
    """
    if run_dir is None:
        run_dir = RUN_DIR
    path = _lib_save_results_csv(
        stage_results_list,
        stage_configs=STAGE_CONFIGS,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        run_dir=run_dir,
    )
    print(f"\nResults CSV saved to: {path}")
    return path


print(f"Training infrastructure ready. Algorithm: {ALGORITHM}")
print("DiagnosticsCallback enabled: per-component rewards, obs/action stats,")
print("VecNormalize tracking, termination reasons, and plateau detection")
print("will log to TensorBoard.")

### Visualization Functions

In [ ]:
from environments.shared.visualization import (
    plot_diagnostics_graphs as _lib_plot_diagnostics_graphs,
    plot_training_curves as _lib_plot_training_curves,
)


def plot_training_curves(stage_dirs, stage_configs, algo_name, save_path=None):
    """Plot evaluation reward, episode length, tilt angle, and forward velocity curves.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global and calls ``plt.show()`` for inline display.
    """
    fig = _lib_plot_training_curves(
        stage_dirs, stage_configs, species=SPECIES, algorithm=algo_name,
        save_path=save_path,
    )
    if save_path is not None:
        print(f"Training curves saved to: {save_path}")
    plt.show()


def plot_diagnostics_graphs(stage_dirs, stage_configs, algo_name, save_dir=None, _show=True):
    """Create diagnostic figures for locomotion health and behavioral metrics.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global.  When ``_show`` is True (default), figures
    are displayed inline; otherwise they are closed after saving.
    """
    fig1, fig2 = _lib_plot_diagnostics_graphs(
        stage_dirs, stage_configs, species=SPECIES, algorithm=algo_name,
        save_dir=save_dir, show=_show,
    )
    if _show:
        plt.show()


print("Visualization functions ready (shared library).")

In [ ]:
# ===== SELECT VALIDATION SETTING (1, 2, 3, or 4) =====
VALIDATION_SETTING = 4  # Set to None to use default TOML config

# Load species-specific validation configs from TOML.
# Each species has its own configs/<species>/sweep_validation.toml with
# species-appropriate env kwargs (e.g. bite_bonus for trex, strike_bonus
# for velociraptor).
if VALIDATION_SETTING is not None:
    try:
        import tomllib
    except ImportError:
        import tomli as tomllib

    _SWEEP_TOML = repo_root / "configs" / SPECIES / "sweep_validation.toml"
    if not _SWEEP_TOML.exists():
        print(
            f"WARNING: VALIDATION_SETTING={VALIDATION_SETTING} ignored — "
            f"no sweep_validation.toml found for {SPECIES}. "
            f"Using default TOML config."
        )
        VALIDATION_SETTING = None

if VALIDATION_SETTING is not None:
    with open(_SWEEP_TOML, "rb") as _f:
        _sweep_raw = tomllib.load(_f)

    # Select algorithm-appropriate hyperparameter section
    _algo_lower = ALGORITHM.lower()
    _algo_key = f"{_algo_lower}_kwargs"

    # Discover available settings dynamically (setting_1, setting_2, etc.)
    _VALIDATION_CONFIGS = {}
    for _key, _section in _sweep_raw.items():
        if _key.startswith("setting_") and isinstance(_section, dict) and "env" in _section:
            _n = int(_key.split("_")[1])
            _env = dict(_section["env"])
            # Read the algorithm-specific section (ppo or sac)
            if _algo_lower not in _section:
                print(
                    f"WARNING: setting_{_n} has no [{_key}.{_algo_lower}] section — skipping. "
                    f"Available: {[k for k in _section if k != 'env']}"
                )
                continue
            _algo_params = dict(_section[_algo_lower])
            # Convert list ranges to tuples (same as load_stage_config does)
            for _k, _v in _env.items():
                if isinstance(_v, list):
                    _env[_k] = tuple(_v)
            _VALIDATION_CONFIGS[_n] = {"env_kwargs": _env, _algo_key: _algo_params}

    assert VALIDATION_SETTING in _VALIDATION_CONFIGS, (
        f"Invalid VALIDATION_SETTING={VALIDATION_SETTING} for {SPECIES} ({ALGORITHM}). "
        f"Available: {sorted(_VALIDATION_CONFIGS.keys())}"
    )
    _selected = _VALIDATION_CONFIGS[VALIDATION_SETTING]

    # Override env_kwargs (merge to preserve any keys from base TOML)
    STAGE_CONFIGS[1]["env_kwargs"].update(_selected["env_kwargs"])

    # Override algorithm kwargs (merge to preserve any keys not in sweep)
    STAGE_CONFIGS[1][_algo_key].update(_selected[_algo_key])

    # Propagate net_arch to stages 2 and 3 so all stages use the same
    # policy architecture.  Without this, loading a stage 1 checkpoint
    # with e.g. [512, 256] into a stage 2 model with [256, 256] silently
    # discards half the first-layer neurons.
    _s1_net_arch = _selected[_algo_key].get("policy_kwargs", {}).get("net_arch")
    if _s1_net_arch is not None:
        for _stage_num in (2, 3):
            if _stage_num in STAGE_CONFIGS:
                STAGE_CONFIGS[_stage_num][_algo_key].setdefault("policy_kwargs", {})["net_arch"] = list(_s1_net_arch)
        print(f"  net_arch {_s1_net_arch} propagated to stages 2 and 3")

    _algo_params = _selected[_algo_key]
    print(f"Validation Setting {VALIDATION_SETTING} applied to Stage 1 config ({SPECIES}, {ALGORITHM}).")
    print(f"  Config file: {_SWEEP_TOML}")
    print(f"  alive_bonus:          {_selected['env_kwargs'].get('alive_bonus', 'N/A')}")
    print(f"  energy_penalty_weight:{_selected['env_kwargs'].get('energy_penalty_weight', 'N/A')}")
    print(f"  nosedive_weight:      {_selected['env_kwargs'].get('nosedive_weight', 'N/A')}")
    print(f"  posture_weight:       {_selected['env_kwargs'].get('posture_weight', 'N/A')}")
    if "drift_penalty_weight" in _selected["env_kwargs"]:
        print(f"  drift_penalty_weight: {_selected['env_kwargs']['drift_penalty_weight']}")
    if "heading_weight" in _selected["env_kwargs"]:
        print(f"  heading_weight:       {_selected['env_kwargs']['heading_weight']}")
    if "spin_penalty_weight" in _selected["env_kwargs"]:
        print(f"  spin_penalty_weight:  {_selected['env_kwargs']['spin_penalty_weight']}")
    print(f"  learning_rate:        {_algo_params['learning_rate']:.2e}" if isinstance(_algo_params['learning_rate'], float) else f"  learning_rate:        {_algo_params['learning_rate']}")
    print(f"  batch_size:           {_algo_params['batch_size']}")
    print(f"  gamma:                {_algo_params['gamma']:.6f}")
    if _algo_lower == "ppo":
        print(f"  n_epochs:             {_algo_params['n_epochs']}")
        print(f"  n_steps:              {_algo_params['n_steps']}")
        print(f"  ent_coef:             {_algo_params['ent_coef']:.6e}")
    else:
        print(f"  tau:                  {_algo_params.get('tau', 'N/A')}")
        print(f"  ent_coef:             {_algo_params.get('ent_coef', 'N/A')}")
        print(f"  buffer_size:          {_algo_params.get('buffer_size', 'N/A')}")
    print(f"  net_arch:             {_algo_params.get('policy_kwargs', {}).get('net_arch', 'N/A')}")
else:
    print("Using default TOML config for Stage 1 (no validation override).")

## Validation Settings Override

Select one of the hyperparameter configurations to validate.
Validation configs are loaded from `configs/<species>/sweep_validation.toml`,
so each species has its own species-appropriate settings.

**Velociraptor** (`configs/velociraptor/sweep_validation.toml`):

| Setting | Source | Best Reward | Ep Length | Key Traits |
|---------|--------|-------------|-----------|------------|
| 1 | Sweep Trial 20 | 4516.53 | 972 | Highest reward, perfectly stable |
| 2 | Sweep Trial 9 | 4383.09 | 1000 | Max episode length, very stable |
| 3 | Sweep Trial 21 | 4380.45 | 1000 | Lowest LR, max episode length |
| 4 | Manual (anti-rotation) | TBD | TBD | Reduced alive bonus, heading/spin |

**T-Rex** (`configs/trex/sweep_validation.toml`):

| Setting | Source | Best Reward | Ep Length | Key Traits |
|---------|--------|-------------|-----------|------------|
| 1 | Baseline (TOML) | TBD | TBD | Default stage1_balance.toml values |
| 2 | Anti-rotation | TBD | TBD | Stronger heading/spin penalties |
| 3 | High alive bonus | TBD | TBD | Boosted alive bonus, tuned posture |
| 4 | Raptor S4 port | TBD | TBD | Velociraptor setting 4 (anti-rot.) |

Set `VALIDATION_SETTING` in the cell above to select a setting number.
Set to `None` to skip the override and use the default TOML config.

## 5. Stage 1: Balance

The raptor learns to stand upright without falling. No forward velocity reward —
just a strong alive bonus.

In [ ]:
cur1 = STAGE_CONFIGS[1].get("curriculum_kwargs", {})
timesteps_1 = 50_000 if QUICK_TEST else cur1.get("timesteps", 6_000_000)

model_1, path_1, final_path_1, dir_1, vecnorm_1, results_1 = train_stage(stage=1, timesteps=timesteps_1, run_dir=RUN_DIR)

In [ ]:
# Generate all stage artifacts: summary, videos, and graphs (saved to disk)
generate_stage_artifacts(
    species_cfg=SPECIES_CFG,
    stage_config=STAGE_CONFIGS[1],
    stage=1,
    algorithm=ALGORITHM,
    stage_dir=dir_1,
    seed=SEED,
    stage_results=results_1,
)

# Display saved videos inline
display_stage_videos(1, dir_1)

# Display training curves and diagnostics inline
completed_stages.append((1, dir_1))
plot_training_curves([(1, dir_1)], STAGE_CONFIGS, ALGORITHM)
plot_diagnostics_graphs([(1, dir_1)], STAGE_CONFIGS, ALGORITHM)

# Export collected_results.csv incrementally (overwrites with all stages so far)
save_results_csv([results_1], species=SPECIES)

# Write training summary with all completed stages so far (before gate check
# so the summary is always generated even if the gate fails)
write_training_summary(RUN_DIR, [results_1])

# Enforce curriculum gate after generating artifacts
if not results_1["gate_passed"]:
    raise RuntimeError(
        "Stage 1 failed curriculum gate: "
        + "; ".join(results_1["gate_failures"])
        + "."
    )

## 6. Stage 2: Locomotion

Starting from the Stage 1 checkpoint, the raptor learns to walk and run forward.

In [ ]:
cur2 = STAGE_CONFIGS[2].get("curriculum_kwargs", {})
timesteps_2 = 50_000 if QUICK_TEST else cur2.get("timesteps", 8_000_000)

model_2, path_2, final_path_2, dir_2, vecnorm_2, results_2 = train_stage(
    stage=2, timesteps=timesteps_2, load_path=path_1, run_dir=RUN_DIR, vecnorm_path=vecnorm_1
)

In [ ]:
# Generate all stage artifacts: summary, videos, and graphs (saved to disk)
generate_stage_artifacts(
    species_cfg=SPECIES_CFG,
    stage_config=STAGE_CONFIGS[2],
    stage=2,
    algorithm=ALGORITHM,
    stage_dir=dir_2,
    seed=SEED,
    stage_results=results_2,
)

# Display saved videos inline
display_stage_videos(2, dir_2)

# Display training curves and diagnostics inline
completed_stages.append((2, dir_2))
plot_training_curves([(2, dir_2)], STAGE_CONFIGS, ALGORITHM)
plot_diagnostics_graphs([(2, dir_2)], STAGE_CONFIGS, ALGORITHM)

# Export collected_results.csv incrementally (overwrites with all stages so far)
save_results_csv([results_1, results_2], species=SPECIES)

# Write training summary with all completed stages so far (before gate check
# so the summary is always generated even if the gate fails)
write_training_summary(RUN_DIR, [results_1, results_2])

# Enforce curriculum gate after generating artifacts
if not results_2["gate_passed"]:
    raise RuntimeError(
        "Stage 2 failed curriculum gate: "
        + "; ".join(results_2["gate_failures"])
        + "."
    )

## 7. Stage 3: Species-Specific Behavior

The final stage enables the species-specific reward (strike/bite/food reach) and trains the full behavioral repertoire. The stage 3 label and description are configured in the species selection cell above.

In [ ]:
cur3 = STAGE_CONFIGS[3].get("curriculum_kwargs", {})
timesteps_3 = 50_000 if QUICK_TEST else cur3.get("timesteps", 8_000_000)

model_3, path_3, final_path_3, dir_3, vecnorm_3, results_3 = train_stage(
    stage=3, timesteps=timesteps_3, load_path=path_2, run_dir=RUN_DIR, vecnorm_path=vecnorm_2
)

In [ ]:
# Generate all stage artifacts: summary, videos, and graphs (saved to disk)
generate_stage_artifacts(
    species_cfg=SPECIES_CFG,
    stage_config=STAGE_CONFIGS[3],
    stage=3,
    algorithm=ALGORITHM,
    stage_dir=dir_3,
    seed=SEED,
    stage_results=results_3,
)

# Display saved videos inline
display_stage_videos(3, dir_3)

# Display training curves and diagnostics inline
completed_stages.append((3, dir_3))
plot_training_curves([(3, dir_3)], STAGE_CONFIGS, ALGORITHM)
plot_diagnostics_graphs([(3, dir_3)], STAGE_CONFIGS, ALGORITHM)

# Export collected_results.csv incrementally (overwrites with all stages so far)
save_results_csv([results_1, results_2, results_3], species=SPECIES)

# Write training summary with all completed stages so far (before gate check
# so the summary is always generated even if the gate fails)
write_training_summary(RUN_DIR, [results_1, results_2, results_3])

# Enforce curriculum gate after generating artifacts
if not results_3["gate_passed"]:
    raise RuntimeError(
        "Stage 3 failed curriculum gate: "
        + "; ".join(results_3["gate_failures"])
        + "."
    )

## 8. Evaluate Final Policy

In [ ]:
# Evaluate the final Stage 3 policy using the library's evaluate() function,
# which provides full locomotion metrics (gait symmetry, cost of transport,
# stride frequency, velocity consistency, etc.) via LocomotionMetrics.
print(f"Evaluating final Stage 3 policy ({ALGORITHM})...")
evaluate(
    species_cfg=SPECIES_CFG,
    stage_configs=STAGE_CONFIGS,
    model_path=path_3 + ".zip",
    n_episodes=30,
    render=False,
    stage=3,
    algorithm=ALGORITHM.lower(),
)

## 9. Training Curves

In [ ]:
# Re-plot individual stage graphs (convenient for re-running this cell standalone)
for stage_num, stage_dir in completed_stages:
    print(f"\n{'=' * 40}")
    print(f"Stage {stage_num}: {STAGE_CONFIGS[stage_num]['name']}")
    print(f"{'=' * 40}")
    plot_training_curves([(stage_num, stage_dir)], STAGE_CONFIGS, ALGORITHM, save_path=Path(stage_dir) / "training_curves.png")
    plot_diagnostics_graphs([(stage_num, stage_dir)], STAGE_CONFIGS, ALGORITHM, save_dir=stage_dir)

## 10. Replay All Stage Videos

Display the saved videos for all completed training stages. Videos are recorded
by `generate_stage_artifacts` after each stage completes.

In [ ]:
for stage_num, stage_dir in completed_stages:
    print(f"\n{'=' * 40}")
    print(f"Stage {stage_num}: {STAGE_CONFIGS[stage_num]['name']}")
    print(f"{'=' * 40}")
    display_stage_videos(stage_num, stage_dir)

## 11. Cleanup

In [ ]:
write_training_summary(RUN_DIR, [results_1, results_2, results_3])
save_results_json([results_1, results_2, results_3], species=SPECIES)
save_results_csv([results_1, results_2, results_3], species=SPECIES)

print("Training complete!")
print(f"\nAlgorithm: {ALGORITHM}")
print(f"Run directory: {RUN_DIR}")
print(f"Stage 1 best model: {path_1}.zip")
print(f"Stage 1 final model: {final_path_1}.zip")
print(f"Stage 2 best model: {path_2}.zip")
print(f"Stage 2 final model: {final_path_2}.zip")
print(f"Stage 3 best model: {path_3}.zip")
print(f"Stage 3 final model: {final_path_3}.zip")
print("\nTo run the other algorithm, change ALGORITHM at the top and re-run all cells.")